# VAHDAM — Post-Purchase Hero Creatives Generator (robust)

Generates the **21 email hero images** with the new **Coffee+** packet swapped in.

**Fixes for blank-file issue:**
- Validates that real image bytes came back (no more 0-byte / blank PNGs)
- Detects all-white / all-transparent (blank) outputs and reports them
- Prints the model's refusal / safety text when no image is returned
- **Auto-falls back to Gemini** if OpenAI returns nothing (and vice-versa)

Run cells top to bottom. Provide **either or both** API keys in Cell 2.

In [ ]:
# ── Cell 1 · Install deps (both backends so fallback works) ────────────────
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
                       "openai>=1.51", "google-genai", "pillow", "requests"])
print("✅ packages ready (openai + google-genai)")

In [ ]:
# ── Cell 2 · API keys — enter EITHER or BOTH (press Enter to skip one) ──────
import os, getpass

k = getpass.getpass("OpenAI API key (Enter to skip): ").strip()
if k: os.environ["OPENAI_API_KEY"] = k

k = getpass.getpass("Gemini API key (Enter to skip): ").strip()
if k: os.environ["GEMINI_API_KEY"] = k

HAS_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
HAS_GEMINI = bool(os.environ.get("GEMINI_API_KEY"))

# Order to try: primary first, then fallback. Gemini is very reliable for
# 'use this product, recreate this scene', so prefer it if available.
BACKEND_ORDER = [b for b, ok in [("gemini", HAS_GEMINI), ("openai", HAS_OPENAI)] if ok]
assert BACKEND_ORDER, "Provide at least one API key!"
print(f"✅ keys set. Backend order: {BACKEND_ORDER}")

In [ ]:
# ── Cell 3 · Upload packet image ───────────────────────────────────────────
import os
from PIL import Image
import IPython.display as display

PACKET_PATH = None
try:
    from google.colab import files as colab_files
    print("Colab detected — upload your new Coffee+ packet image:")
    uploaded = colab_files.upload()
    PACKET_PATH = list(uploaded.keys())[0]
except ImportError:
    PACKET_PATH = "./new-packet.png"   # ← local path

assert os.path.exists(PACKET_PATH), f"Packet not found: {PACKET_PATH}"

# Normalize: convert to clean RGBA PNG, downscale if huge (API limits / cost).
_im = Image.open(PACKET_PATH).convert("RGBA")
if max(_im.size) > 1536:
    _im.thumbnail((1536, 1536))
PACKET_PATH = "packet_clean.png"
_im.save(PACKET_PATH, "PNG")
print(f"✅ packet normalized -> {PACKET_PATH}  size={_im.size}")
_prev = _im.copy(); _prev.thumbnail((350, 350)); display.display(_prev)

In [ ]:
# ── Cell 4 · Load prompts + robust generators with validation & fallback ───
import json, base64, time
from pathlib import Path
import requests as req
from PIL import Image

PROMPTS_URL = (
    "https://raw.githubusercontent.com/anchittandon-vahdam/"
    "vahdam-trustpilot-mailers/claude/charming-noether-DPvq2/"
    "creatives-pipeline/prompts.json"
)
DATA = req.get(PROMPTS_URL, timeout=20).json()
print(f"✅ loaded {len(DATA['creatives'])} prompts")

OUT_DIR = Path("./creatives"); OUT_DIR.mkdir(exist_ok=True)
BROWSER_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0 Safari/537.36",
    "Accept": "image/avif,image/webp,image/png,image/*,*/*;q=0.8",
    "Referer": "https://www.vahdam.co.uk/",
}

def build_prompt(item):
    parts = [
        "TASK: Produce a premium marketing email hero image.",
        f"PRODUCT (use the attached packet image as the exact reference): {DATA['product_reference']}",
        f"SCENE: {item['prompt']}",
        f"STYLE: {DATA['global_style']}",
    ]
    if item.get("role") == "gift_hero_with_pouch_prop":
        parts.append(f"IMPORTANT: Hero is the gift ({item.get('gift_item','the gift')}); pouch is a supporting prop only.")
    else:
        parts.append("IMPORTANT: The Coffee+ pouch is the HERO subject.")
    parts.append("Do NOT show old packaging. Only the new dark forest-green Coffee+ pouch.")
    return "\n".join(parts)

def _src_bytes(url):
    if not url: return None
    try:
        r = req.get(url, headers=BROWSER_HEADERS, timeout=20)
        if r.status_code == 200 and len(r.content) > 2000: return r.content
    except Exception: pass
    return None

def _is_blank(path):
    """True if image is effectively uniform (all white/black/transparent)."""
    try:
        im = Image.open(path).convert("L")
        lo, hi = im.getextrema()
        return (hi - lo) < 8   # almost no tonal variation = blank
    except Exception:
        return True

def _validate(path):
    """Return None if good, else a reason string."""
    p = Path(path)
    if not p.exists() or p.stat().st_size < 1000:
        return f"file missing/too small ({p.stat().st_size if p.exists() else 0} bytes)"
    if _is_blank(path):
        return "image is blank (uniform color)"
    return None

# ---------- OpenAI ----------
def _gen_openai(item):
    from openai import OpenAI
    client = OpenAI()
    imgs = [open(PACKET_PATH, "rb")]
    sb = _src_bytes(item.get("source_url"))
    tmp = None
    if sb:
        tmp = OUT_DIR / f"_s_{item['id']}.png"; tmp.write_bytes(sb); imgs.append(open(tmp, "rb"))
    try:
        resp = client.images.edit(model="gpt-image-1", image=imgs,
                                   prompt=build_prompt(item), size="1536x1024")
        if not resp.data or not getattr(resp.data[0], "b64_json", None):
            return None, "OpenAI returned no image data"
        raw = base64.b64decode(resp.data[0].b64_json)
        if len(raw) < 1000:
            return None, f"OpenAI returned {len(raw)} bytes"
        dest = OUT_DIR / f"{item['id']}.png"; dest.write_bytes(raw)
        bad = _validate(dest)
        return (None, bad) if bad else (dest, None)
    except Exception as e:
        return None, f"OpenAI error: {e}"
    finally:
        for fh in imgs:
            try: fh.close()
            except: pass
        if tmp and tmp.exists(): tmp.unlink()

# ---------- Gemini ----------
def _gen_gemini(item):
    from google import genai
    from google.genai import types
    client = genai.Client()
    contents = [build_prompt(item),
                types.Part.from_bytes(data=Path(PACKET_PATH).read_bytes(), mime_type="image/png")]
    sb = _src_bytes(item.get("source_url"))
    if sb: contents.append(types.Part.from_bytes(data=sb, mime_type="image/png"))
    try:
        resp = client.models.generate_content(model="gemini-2.5-flash-image", contents=contents)
        cand = resp.candidates[0] if resp.candidates else None
        if not cand:
            fb = getattr(resp, "prompt_feedback", None)
            return None, f"Gemini blocked (prompt_feedback={fb})"
        img_bytes, texts = None, []
        for part in cand.content.parts:
            if getattr(part, "inline_data", None) and part.inline_data.data:
                img_bytes = part.inline_data.data
            elif getattr(part, "text", None):
                texts.append(part.text)
        if not img_bytes:
            reason = getattr(cand, "finish_reason", "?")
            msg = " | ".join(texts)[:300] or "no text"
            return None, f"Gemini no image (finish_reason={reason}): {msg}"
        dest = OUT_DIR / f"{item['id']}.png"; dest.write_bytes(img_bytes)
        bad = _validate(dest)
        return (None, bad) if bad else (dest, None)
    except Exception as e:
        return None, f"Gemini error: {e}"

_GEN = {"openai": _gen_openai, "gemini": _gen_gemini}

def generate(item):
    """Try each backend in BACKEND_ORDER until one returns a valid image."""
    print(f"\n[{item['id']}] {item['alt'][:55]}...")
    for backend in BACKEND_ORDER:
        path, err = _GEN[backend](item)
        if path:
            print(f"  ✅ {backend} -> {path}")
            return path
        print(f"  ⚠️ {backend} failed: {err}")
        time.sleep(2)
    print(f"  ❌ ALL backends failed for {item['id']}")
    return None

print("✅ robust generator ready — backends:", BACKEND_ORDER)

In [ ]:
# ── Cell 5 · TEST — PP1 only (validate before full batch) ──────────────────
import IPython.display as display
from PIL import Image

item = next(c for c in DATA["creatives"] if c["id"] == "PP1")
p = generate(item)
if p:
    im = Image.open(p); im.thumbnail((700, 400)); display.display(im)
    print("size:", Image.open(p).size, "| bytes:", p.stat().st_size)
else:
    print("\n⚠️ Still failing. Read the per-backend reason printed above:")
    print("  - 'blocked'/'finish_reason=SAFETY' → prompt tripped safety; tell me and I'll soften it")
    print("  - 'no image data' on OpenAI → your key may lack gpt-image-1 access; use Gemini key")
    print("  - 'billing'/'quota' → top up the account")

In [ ]:
# ── Cell 6 · FULL BATCH (all 21) ──────────────────────────────────────────
results = {}
for c in DATA["creatives"]:
    p = generate(c)
    results[c["id"]] = str(p) if p else "FAILED"

ok  = [k for k, v in results.items() if v != "FAILED"]
err = [k for k, v in results.items() if v == "FAILED"]
print(f"\n=== DONE ===\n✅ {len(ok)}: {ok}")
if err: print(f"❌ {len(err)}: {err}")

In [ ]:
# ── Cell 7 · Preview all ───────────────────────────────────────────────────
import IPython.display as display
from PIL import Image
for png in sorted(OUT_DIR.glob("PP*.png")):
    print(f"\n{png.name}  ({png.stat().st_size} bytes)")
    im = Image.open(png); im.thumbnail((700, 400)); display.display(im)

In [ ]:
# ── Cell 8 · Download all as zip ───────────────────────────────────────────
import zipfile, io
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
    for png in sorted(OUT_DIR.glob("PP*.png")):
        zf.write(png, png.name)
buf.seek(0)
try:
    from google.colab import files as colab_files
    open("vahdam_creatives.zip", "wb").write(buf.read())
    colab_files.download("vahdam_creatives.zip")
except ImportError:
    print(f"✅ images in {OUT_DIR.resolve()}")